In [2]:
%reload_ext autoreload
%autoreload 2
# notebooks/result_processing.ipynb

# ---
# # Experiment Results Analysis
# This notebook orchestrates the loading, analysis, validation, and reporting 
# of experiment results logged to Weights & Biases.
# ---

# ## 1. Setup and Imports
# Import necessary libraries and custom modules. The `sys.path.append` line
# allows the notebook to find the custom `lmetk_report` package.
import sys
import pandas as pd
sys.path.append('..')

from lmetk_report.run_processor import load_runs_df
from lmetk_report.analysis import analyze_results
from lmetk_report.reporting import (
    create_summary_table,
    display_experiment_stats,
    display_performance_tables,
    display_hp_impact
)

# ---
# ## 2. Configuration
# Define all key parameters for the analysis in one place.
# ---
PROJECT_NAME = "28_01_2026_unibo_prognostics_combined"
USER = "imos-buildingenergy-1"

# Metrics to extract
METRIC_PREFIXES = ["val/", "test/", "test_best_rerun/", "val_best_rerun/"]
REPORTING_METRICS = ["loss", "mse", "mae", "rmse", "f1", "accuracy"]

# --- NEW: Scientific Rigor Settings ---
# If defined, the analysis will DROP any configuration group that does not 
# contain exactly these seeds. Set to `None` to include all available data.

# Data seeds are set to None (Flexible mode)
REQUIRED_DATA_SEEDS = None      

# Model seeds are strictly enforced (Strict mode)
REQUIRED_MODEL_SEEDS = {72, 88, 101, 666, 226688}

# Columns to ignore if they vary within a group (avoids "Hidden Variation" warnings)
ADDITIONAL_IGNORED_COLS = ["gpu_id", "slurm_job_id"] 

# Configure pandas display options for better readability in the notebook.
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# ---
# ## 3. Data Loading
# Fetch data from wandb (or a local cache) and prepare the initial DataFrame.
# The `load_runs_df` function handles all the complex cleaning and flattening.
# ---
print("🚀 Loading Run Data...")
df, config_columns, dropped_columns = load_runs_df(
    project_name=PROJECT_NAME, 
    user=USER
)
print(f"Loaded {len(df)} runs.")

# ---
# ## 4. Core Analysis
# Run the main analysis function. This now includes:
# 1. Grouping by Model + Hyperparameters
# 2. Strict Seed Validation (Filtering incomplete groups)
# 3. Hidden Variation Detection (Warning about silent bugs)
# 4. Aggregation & Best Model Selection
# ---
print("\n🔬 Analyzing Results & Validating Consistency...")

all_results = analyze_results(
    df=df,
    config_columns=config_columns,
    dropped_columns=dropped_columns,
    reporting_metrics=REPORTING_METRICS,
    metric_prefixes=METRIC_PREFIXES,
    # --- New Validation Arguments ---
    required_data_seeds=REQUIRED_DATA_SEEDS,
    required_model_seeds=REQUIRED_MODEL_SEEDS,
    additional_ignored_cols=ADDITIONAL_IGNORED_COLS
)

# ---
# ## 5. Reporting & Visualizations
# ---

# ### A. Experiment Statistics
# Shows how many configurations completed successfully and 
# confirms which seeds were actually found in the data.
display_experiment_stats(all_results)

# ### B. Performance Leaderboards
# Displays a separate, clean pivot table for every metric found (MSE, MAE, etc.).
display_performance_tables(all_results)

# ### C. Hyperparameter Impact Analysis
# Shows the top 10 configurations for each model, exposing exactly 
# which hyperparameters led to the best results.
display_hp_impact(all_results)

# ### D. Legacy Summary Table
# A single flattened table containing all metrics (useful for copying to Excel/LaTeX).
print("\n📊 Full Summary Table:")
summary_df = create_summary_table(all_results)
display(summary_df)

🚀 Loading Run Data...
Loaded 120 runs from cached file: csv_files/28_01_2026_unibo_prognostics_combined.csv
Dropped 22 columns with >90% NaN values.
The columns were drropped: ['ckpt_path', 'pad_value', 'null_value', 'target_mask', 'profiler_cfg', 'subset_range', 'val_subset_ratio', 'optimizer_factory', 'scheduler_factory', 'test_subset_ratio', 'variable_emb_type', 'datamodule.val_subset_ratio', 'datamodule.test_subset_ratio', 'callbacks.early_stopping.stopping_threshold', 'callbacks.early_stopping.divergence_threshold', 'callbacks.early_stopping.check_on_train_epoch_end', 'callbacks.model_checkpoint.every_n_epochs', 'callbacks.model_checkpoint.every_n_train_steps', 'callbacks.model_checkpoint.train_time_interval', 'callbacks.model_checkpoint.save_on_train_epoch_end', 'evaluator._evaluator_defaults.inverse_transform_name', 'logger.wandb.anonymous']
Dropped 60 columns based on name filters (e.g., 'paths.').
Loaded 120 runs.

🔬 Analyzing Results & Validating Consistency...
Dynamic metric

,Dataset,Model,Configs Completed,Seeds Found
0,UNIBO21,lmetk.baselines.spacetimeformer_model.Spacetimeformer_Forecaster,24,Data: ? | Model: ?



  METRIC: test/loss


Dataset,UNIBO21
Model,
lmetk.baselines.spacetimeformer_model.Spacetimeformer_Forecaster,0.02 ± 0.00



  METRIC: test/mae_denormalized_mean


...

Dataset,UNIBO21
Model,
lmetk.baselines.spacetimeformer_model.Spacetimeformer_Forecaster,0.07 ± nan



  HYPERPARAMETER IMPACT ANALYSIS

>>> lmetk.baselines.spacetimeformer_model.Spacetimeformer_Forecaster on UNIBO21
    (Sorted by val/loss)


,optimization.lr,task_definition.seq_len,datamodule.train_batch_size,test/mse_denormalized_mean_mean,val/phm_score_normalized_mean_mean,test/phm_score_normalized_mean_mean,test_best_rerun/loss_mean,val_best_rerun/rmse_denormalized_mean_mean,test_best_rerun/mse_normalized_mean_mean,val/mape_normalized_mean_mean,test/phm_score_denormalized_mean_mean,test_best_rerun/phm_score_normalized_mean_mean,val/phm_score_denormalized_mean_mean,val_best_rerun/mape_normalized_mean_mean,val_best_rerun/mse_normalized_mean_mean,val/loss_mean,val/loss_std,val/loss_count,val_best_rerun/mae_denormalized_mean_mean,test/mae_normalized_mean_mean,test_best_rerun/rmse_normalized_mean_mean,val/mse_denormalized_mean_mean,test_best_rerun/rmse_denormalized_mean_mean,val/rmse_denormalized_mean_mean,test_best_rerun/mae_normalized_mean_mean,val/mape_denormalized_mean_mean,val_best_rerun/mse_denormalized_mean_mean,val_best_rerun/phm_score_denormalized_mean_mean,test/loss_mean,val/mae_normalized_mean_mean,val_best_rerun/mae_normalized_mean_mean,test/mse_normalized_mean_mean,test/mae_denormalized_mean_mean,val_best_rerun/mape_denormalized_mean_mean,val_best_rerun/rmse_normalized_mean_mean,test_best_rerun/mape_normalized_mean_mean,val_best_rerun/loss_mean,test_best_rerun/phm_score_denormalized_mean_mean,test/rmse_denormalized_mean_mean,test/mape_normalized_mean_mean,test_best_rerun/mae_denormalized_mean_mean,val/mse_normalized_mean_mean,val/mae_denormalized_mean_mean,test/mape_denormalized_mean_mean,val/rmse_normalized_mean_mean,test_best_rerun/mape_denormalized_mean_mean,val_best_rerun/phm_score_normalized_mean_mean,test/rmse_normalized_mean_mean,test_best_rerun/mse_denormalized_mean_mean
0,0.0001,1,16,7513.618359,0.110676,0.102013,0.023115,35.014511,0.031007,0.837004,0.102013,0.102013,0.110676,0.904837,0.006123,0.011740,0.001957,5,25.778437,0.114929,0.137826,2126.277490,67.845916,41.137372,0.114929,0.837004,1483.823242,0.118758,0.023115,0.060379,0.052368,0.031007,56.574957,0.904837,0.071130,2.114561,0.006590,0.102013,67.845916,2.114561,56.574957,0.008775,29.722081,2.114561,0.083569,2.114561,0.118758,0.137826,7513.618359
9,0.0005,1,512,5298.639209,0.087477,0.114507,0.024087,42.551754,0.021866,1.005984,0.114507,0.114507,0.087477,1.064650,0.008534,0.012950,0.002051,5,32.416226,0.098232,0.122403,2509.018359,60.253879,44.674936,0.098232,1.005984,2068.035583,0.103901,0.024087,0.068851,0.065852,0.021866,48.355395,1.064650,0.086442,1.378701,0.010465,0.114507,60.253879,1.378701,48.355395,0.010354,33.892587,1.378701,0.090755,1.378701,0.103901,0.122403,5298.639209
11,0.0005,10,512,3295.151904,0.091675,0.140324,0.014727,40.158039,0.013598,0.738109,0.140324,0.140324,0.091675,0.874674,0.007580,0.013267,0.003168,5,29.549261,0.072107,0.089804,2428.470313,44.206833,40.728896,0.072107,0.738109,1836.695679,0.065996,0.014727,0.061468,0.060028,0.013598,35.495135,0.874674,0.081579,1.190651,0.009420,0.140324,44.206833,1.190651,35.495135,0.010022,30.257942,1.190651,0.082739,1.190651,0.065996,0.089804,3295.151904
8,0.0005,1,16,6845.953516,0.091589,0.091232,0.027548,47.586044,0.028252,1.116033,0.091232,0.091232,0.091589,1.102809,0.010335,0.013474,0.003658,5,34.671883,0.112254,0.139229,2778.544067,68.536530,48.362312,0.112254,1.116033,2504.326660,0.086456,0.027548,0.073231,0.070434,0.028252,55.257707,1.102809,0.096669,1.655881,0.012910,0.091232,68.536530,1.655881,55.257707,0.011467,36.048388,1.655881,0.098246,1.655881,0.086456,0.139229,6845.953516
1,0.0001,1,512,4169.912598,0.082585,0.116917,0.018176,NaN,0.017208,1.113755,0.116917,0.116917,0.082585,NaN,NaN,0.014875,0.003201,5,NaN,0.097144,0.116773,2906.165234,57.482666,48.319426,0.097144,1.113755,NaN,NaN,0.018176,0.077770,NaN,0.017208,47.820101,NaN,NaN,1.619307,NaN,0.116917,57.482666,1.619307,47.820101,0.011993,38.282922,1.619307,0.098159,1.619307,NaN,0.116773,4169.912598
17,0.0010,1,512,4006.444727,0.074386,0.117609,0.017538,NaN,0.016534,1.398732,0.117609,0.117609,0.074386,NaN,NaN,0.015746,0.004994,5,NaN,0.090664,0.109814,3290.237646,54.


📊 Full Summary Table:


Dataset                                                           UNIBO21                                                                                                            \
Metric                                                          test/loss test/mae_denormalized_mean test/mae_normalized_mean test/mape_denormalized_mean test/mape_normalized_mean   
Model                                                                                                                                                                                 
lmetk.baselines.spacetimeformer_model.Spacetime...  0.0231 ± 0.0035 (n=5)     56.5750 ± 4.7918 (n=5)    0.1149 ± 0.0097 (n=5)       2.1146 ± 0.2343 (n=5)     2.1146 ± 0.2343 (n=5)   

Dataset                                                                                                                                                                                               \
Metric                                               test/mse_denormalized_mean test/mse_normalized_mean test/phm_score_denormalized_mean test/phm_score_normalized_mean test/rmse_denormalized_mean   
Model                                                                                                                                                                                                  
lmetk.baselines.spacetimeformer_model.Spacetime...  7513.6184 ± 2080.1085 (n=5)    0.0310 ± 0.0086 (n=5)            0.1020 ± 0.0145 (n=5)          0.1020 ± 0.0145 (n=5)      67.8459 ± 5.0170 (n=5)   

Dataset                                                                                                                                                                        \
Metric                                             test/rmse_normalized_mean   test_best_rerun/loss test_best_rerun/mae_denormalized_mean test_best_rerun/mae_normalized_mean   
Model                                                                                                                                                                           
lmetk.baselines.spacetimeformer_model.Spacetime...     0.1378 ± 0.0102 (n=5)  0.0231 ± 0.0035 (n=5)                56.5750 ± 4.7918 (n=5)               0.1149 ± 0.0097 (n=5)   

Dataset                                                                                                                                                               \
Metric                                             test_best_rerun/mape_denormalized_mean test_best_rerun/mape_normalized_mean test_best_rerun/mse_denormalized_mean   
Model                                                                                                                                                                  
lmetk.baselines.spacetimeformer_model.Spacetime...                  2.1146 ± 0.2343 (n=5)                2.1146 ± 0.2343 (n=5)           7513.6184 ± 2080.1085 (n=5)   

Dataset                                                                                                                                                                       \
Metric                                             test_best_rerun/mse_normalized_mean test_best_rerun/phm_score_denormalized_mean test_best_rerun/phm_score_normalized_mean   
Model                                                                                                                                                                          
lmetk.baselines.spacetimeformer_model.Spacetime...               0.0310 ± 0.0086 (n=5)                       0.1020 ± 0.0145 (n=5)                     0.1020 ± 0.0145 (n=5)   

Dataset                                                                                                                                                                          \
Metric                                             test_best_rerun/rmse_denormalized_mean test_best_rerun/rmse_normalized_mean               val/loss val/mae_denormalized_mean   
Model                

In [18]:
# 1. Access the raw data for the specific model and dataset
model_key = 'lmetk.baselines.spacetimeformer_model.Spacetimeformer_Forecaster'
dataset_key = 'UNIBO21'

# This DF contains all runs for this model before aggregation
df_subset = all_results[dataset_key][model_key]['non_aggregated_df']

# 2. Filter for ONE specific configuration that shows count=15
# (Based on your screenshot: lr=0.0001, seq_len=10, batch_size=16)
mask = (
    (df_subset['optimization.lr'] == 0.0001) & 
    (df_subset['task_definition.seq_len'] == 10) & 
    (df_subset['datamodule.train_batch_size'] == 16)
)
group_runs = df_subset[mask]

# 3. Display identifiers to spot the duplicates
# 'created_at' will show you if these runs are from different times/days
cols_to_inspect = ['run_name', 'id', 'created_at', 'seed', 'datasource.parameters.data_seed']
valid_cols = [c for c in cols_to_inspect if c in group_runs.columns]

print(f"Found {len(group_runs)} runs for this configuration (Expected 5).")
print("Listing all runs in this group:")
#display(group_runs[valid_cols].sort_values('created_at'))

KeyError: 'non_aggregated_df'

In [19]:
import pandas as pd

# 1. Isolate the specific group
target_model = 'lmetk.baselines.spacetimeformer_model.Spacetimeformer_Forecaster'

# Update this mask to match the specific "duplicate" group you saw in your table
mask = (
    (df['model._target_'] == target_model) &
    (df['optimization.lr'] == 0.0001) & 
    (df['task_definition.seq_len'] == 10) & 
    (df['datamodule.train_batch_size'] == 16)
)
group_runs = df[mask].copy()

# 2. Find "Hidden Variables" (Robust to lists/dicts)
varying_cols = []

for col in group_runs.columns:
    try:
        # Standard check for hashable types (int, float, str)
        if group_runs[col].nunique() > 1:
            varying_cols.append(col)
    except TypeError:
        # Fallback for unhashable types (lists, dicts): Convert to string first
        if group_runs[col].astype(str).nunique() > 1:
            varying_cols.append(col)

# 3. Add identifiers for context
# 'created_at' or '_timestamp' is CRITICAL to see if these are old vs new runs
context_cols = ['run_name', 'seed', 'created_at', '_timestamp'] 

for c in context_cols:
    if c in group_runs.columns and c not in varying_cols:
        varying_cols.insert(0, c)

# 4. Display sorted by Seed to compare side-by-side
print(f"Found {len(group_runs)} runs in this group.")
print("Showing ONLY columns that differ between these runs:")

if not varying_cols:
    print("No varying columns found! These rows are EXACT duplicates.")
else:
    display(
        group_runs[varying_cols]
        .sort_values(by=['seed', 'created_at'] if 'created_at' in group_runs else ['seed'])
        .style.set_properties(**{'text-align': 'left'})
    )

Found 30 runs in this group.
Showing ONLY columns that differ between these runs:


In [20]:
group_runs.sort_values(by=['seed', 'created_at'] if 'created_at' in group_runs else ['seed'])['val_best_rerun/rmse_denormalized_mean']

25     203.011566
145    202.999359
295           NaN
265           NaN
469           NaN
415           NaN
146           NaN
266           NaN
296           NaN
416     58.815029
26            NaN
470           NaN
27            NaN
267           NaN
471           NaN
147    123.279083
417           NaN
297           NaN
298           NaN
418    223.292770
28     223.274277
268           NaN
472    223.285019
148           NaN
269           NaN
149           NaN
29     172.733551
299           NaN
419           NaN
473    172.723923
Name: val_best_rerun/rmse_denormalized_mean, dtype: float64

In [21]:
group_runs.sort_values(by=['seed', 'created_at'] if 'created_at' in group_runs else ['seed'])[[i for i in group_runs.columns if 'val_best_rerun' in i]]

,val_best_rerun/loss,val_best_rerun/mae_15_denormalized,val_best_rerun/mae_15_normalized,val_best_rerun/mae_18_denormalized,val_best_rerun/mae_18_normalized,val_best_rerun/mae_23_denormalized,val_best_rerun/mae_23_normalized,val_best_rerun/mae_27_denormalized,val_best_rerun/mae_27_normalized,val_best_rerun/mae_5_denormalized,val_best_rerun/mae_5_normalized,val_best_rerun/mae_7_denormalized,val_best_rerun/mae_7_normalized,val_best_rerun/mae_denormalized_mean,val_best_rerun/mae_normalized_mean,val_best_rerun/mape_15_denormalized,val_best_rerun/mape_15_normalized,val_best_rerun/mape_18_denormalized,val_best_rerun/mape_18_normalized,val_best_rerun/mape_23_denormalized,val_best_rerun/mape_23_normalized,val_best_rerun/mape_27_denormalized,val_best_rerun/mape_27_normalized,val_best_rerun/mape_5_denormalized,val_best_rerun/mape_5_normalized,val_best_rerun/mape_7_denormalized,val_best_rerun/mape_7_normalized,val_best_rerun/mape_denormalized_mean,val_best_rerun/mape_normalized_mean,val_best_rerun/mse_15_denormalized,val_best_rerun/mse_15_normalized,val_best_rerun/mse_18_denormalized,val_best_rerun/mse_18_normalized,val_best_rerun/mse_23_denormalized,val_best_rerun/mse_23_normalized,val_best_rerun/mse_27_denormalized,val_best_rerun/mse_27_normalized,val_best_rerun/mse_5_denormalized,val_best_rerun/mse_5_normalized,val_best_rerun/mse_7_denormalized,val_best_rerun/mse_7_normalized,val_best_rerun/mse_denormalized_mean,val_best_rerun/mse_normalized_mean,val_best_rerun/phm_score_15_denormalized,val_best_rerun/phm_score_15_normalized,val_best_rerun/phm_score_18_denormalized,val_best_rerun/phm_score_18_normalized,val_best_rerun/phm_score_23_denormalized,val_best_rerun/phm_score_23_normalized,val_best_rerun/phm_score_27_denormalized,val_best_rerun/phm_score_27_normalized,val_best_rerun/phm_score_5_denormalized,val_best_rerun/phm_score_5_normalized,val_best_rerun/phm_score_7_denormalized,val_best_rerun/phm_score_7_normalized,val_best_rerun/phm_score_denormalized_mean,val_best_rerun/phm_score_normalized_mean,val_best_rerun/rmse_15_denormalized,val_best_rerun/rmse_15_normalized,val_best_rerun/rmse_18_denormalized,val_best_rerun/rmse_18_normalized,val_best_rerun/rmse_23_denormalized,val_best_rerun/rmse_23_normalized,val_best_rerun/rmse_27_denormalized,val_best_rerun/rmse_27_normalized,val_best_rerun/rmse_5_denormalized,val_best_rerun/rmse_5_normalized,val_best_rerun/rmse_7_denormalized,val_best_rerun/rmse_7_normalized,val_best_rerun/rmse_denormalized_mean,val_best_rerun/rmse_normalized_mean
25,0.108939,39.575298,0.036534,150.088043,0.138556,94.722992,0.087445,98.587669,0.091012,85.895142,0.079295,75.829376,0.070003,90.783089,0.083808,0.854746,0.854746,2.177707,2.177707,4.385330,4.385330,6.260159,6.260159,5.324037,5.324037,2.329579,2.329579,3.555259,3.555259,3521.164307,0.003001,545199.750000,0.464634,12407.972656,0.010574,13711.643555,0.011685,9418.506836,0.008027,8990.218750,0.007662,98874.875000,0.084264,0.401498,0.401498,0.281716,0.281716,0.206057,0.206057,0.209837,0.209837,0.227732,0.227732,0.225626,0.225626,0.258744,0.258744,59.339401,0.054780,738.376465,0.681641,111.391083,0.102832,117.096725,0.108099,97.048988,0.089592,94.816765,0.087531,203.011566,0.187413
145,0.108926,39.590000,0.036548,150.091812,0.138559,94.702881,0.087426,98.570015,0.090996,85.885071,0.079286,75.826904,0.070001,90.777779,0.083803,0.853150,0.853150,2.179210,2.179210,4.384621,4.384621,6.258992,6.258991,5.321785,5.321785,2.328809,2.328809,3.554428,3.554428,3523.446777,0.003003,545136.312500,0.464581,12402.447266,0.010570,13706.332031,0.011681,9416.600586,0.008025,8991.702148,0.007663,98862.804688,0.084254,0.401508,0.401508,0.281956,0.281956,0.206136,0.206135,0.209899,0.209899,0.227732,0.227732,0.225661,0.225661,0.258815,0.258815,59.358631,0.054798,738.333435,0.681601,111.366272,0.102809,117.074043,0.108078,97.039169,0.089583,94.824585,0.087538,202.999359,0.187401
295,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [22]:
group_runs.sort_values(by=['seed', 'created_at'] if 'created_at' in group_runs else ['seed'])[[i for i in group_runs.columns if 'test_best_rerun' in i]]

,test_best_rerun/loss,test_best_rerun/mae_16_denormalized,test_best_rerun/mae_16_normalized,test_best_rerun/mae_19_denormalized,test_best_rerun/mae_19_normalized,test_best_rerun/mae_24_denormalized,test_best_rerun/mae_24_normalized,test_best_rerun/mae_28_denormalized,test_best_rerun/mae_28_normalized,test_best_rerun/mae_6_denormalized,test_best_rerun/mae_6_normalized,test_best_rerun/mae_8_denormalized,test_best_rerun/mae_8_normalized,test_best_rerun/mae_denormalized_mean,test_best_rerun/mae_normalized_mean,test_best_rerun/mape_16_denormalized,test_best_rerun/mape_16_normalized,test_best_rerun/mape_19_denormalized,test_best_rerun/mape_19_normalized,test_best_rerun/mape_24_denormalized,test_best_rerun/mape_24_normalized,test_best_rerun/mape_28_denormalized,test_best_rerun/mape_28_normalized,test_best_rerun/mape_6_denormalized,test_best_rerun/mape_6_normalized,test_best_rerun/mape_8_denormalized,test_best_rerun/mape_8_normalized,test_best_rerun/mape_denormalized_mean,test_best_rerun/mape_normalized_mean,test_best_rerun/mse_16_denormalized,test_best_rerun/mse_16_normalized,test_best_rerun/mse_19_denormalized,test_best_rerun/mse_19_normalized,test_best_rerun/mse_24_denormalized,test_best_rerun/mse_24_normalized,test_best_rerun/mse_28_denormalized,test_best_rerun/mse_28_normalized,test_best_rerun/mse_6_denormalized,test_best_rerun/mse_6_normalized,test_best_rerun/mse_8_denormalized,test_best_rerun/mse_8_normalized,test_best_rerun/mse_denormalized_mean,test_best_rerun/mse_normalized_mean,test_best_rerun/phm_score_16_denormalized,test_best_rerun/phm_score_16_normalized,test_best_rerun/phm_score_19_denormalized,test_best_rerun/phm_score_19_normalized,test_best_rerun/phm_score_24_denormalized,test_best_rerun/phm_score_24_normalized,test_best_rerun/phm_score_28_denormalized,test_best_rerun/phm_score_28_normalized,test_best_rerun/phm_score_6_denormalized,test_best_rerun/phm_score_6_normalized,test_best_rerun/phm_score_8_denormalized,test_best_rerun/phm_score_8_normalized,test_best_rerun/phm_score_denormalized_mean,test_best_rerun/phm_score_normalized_mean,test_best_rerun/rmse_16_denormalized,test_best_rerun/rmse_16_normalized,test_best_rerun/rmse_19_denormalized,test_best_rerun/rmse_19_normalized,test_best_rerun/rmse_24_denormalized,test_best_rerun/rmse_24_normalized,test_best_rerun/rmse_28_denormalized,test_best_rerun/rmse_28_normalized,test_best_rerun/rmse_6_denormalized,test_best_rerun/rmse_6_normalized,test_best_rerun/rmse_8_denormalized,test_best_rerun/rmse_8_normalized,test_best_rerun/rmse_denormalized_mean,test_best_rerun/rmse_normalized_mean
25,0.006279,36.599907,0.033788,45.280937,0.041802,91.593674,0.084556,93.346214,0.086174,124.012848,0.114484,64.398254,0.059450,75.871971,0.070042,0.798017,0.798017,3.872612,3.872612,4.559904,4.559904,5.607407,5.607407,8.340112,8.340112,1.667811,1.667811,4.140977,4.140977,2287.628418,0.001950,3568.658691,0.003041,11193.296875,0.009539,12341.879883,0.010518,18358.068359,0.015645,7040.865234,0.006000,9131.732422,0.007782,0.233448,0.233448,0.283971,0.283971,0.201639,0.201639,0.199628,0.199628,0.037096,0.037096,0.270562,0.270562,0.204391,0.204391,47.829159,0.044154,59.738251,0.055148,105.798378,0.097669,111.094009,0.102558,135.491943,0.125081,83.909866,0.077462,90.643600,0.083679
145,0.006278,36.593113,0.033781,45.298717,0.041818,91.576347,0.084540,93.330368,0.086159,123.975784,0.114450,64.400124,0.059452,75.862404,0.070033,0.798193,0.798193,3.878501,3.878501,4.559038,4.559038,5.606248,5.606247,8.338149,8.338149,1.665764,1.665764,4.140983,4.140982,2287.459961,0.001949,3571.055908,0.003043,11188.654297,0.009535,12337.326172,0.010514,18348.730469,0.015637,7042.513184,0.006002,9129.290039,0.007780,0.233551,0.233551,0.283994,0.283994,0.201644,0.201644,0.199665,0.199665,0.037155,0.037155,0.270579,0.270579,0.204431,0.204431,47.827400,0.044152,59.758312,0.055167,105.776436,0.097649,111.073517,0.102539,135.457489,0.125049,83.919685,0.077471,90.635475,0.083671
295,0.006279,36.601372,0.033789,45.279438,0.041